In [18]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.animation as animation

In [19]:
# KITTI-360 image folder
img_dir = r"/home/abid/binta/KITTI_360/download_2d_perspective/KITTI-360/data_2d_raw/2013_05_28_drive_0000_sync/image_00/data_rect"


In [20]:
os.chdir("/home/abid/binta/walkability_abid/")

df_merge = pd.read_csv("outputs/df_merge_redfin.csv")

df_merge["quad_id"] = df_merge["quad_class"].astype(str).str.extract(r"^([A-D])")[0]

quad_grouped = (
    df_merge.dropna(subset=["quad_id"])
    .groupby("quad_id")["frame_ids"]
    .apply(list)
    .reset_index(name="frame_list")
)

quad_grouped

,quad_id,frame_list
0,A,[8585;8586;8587;8588;8589;8590;8591;8622;8623;...
1,B,[8581;8582;8583;8584;8585;8586;8587;8588;8589;...
2,C,[10262;10263;10264;10265;10266;10267;10268;102...
3,D,[10268;10269;10271;10272;10273;10274;10275;102...


In [21]:
quad_labels = {
    "A": "High Walk / Low Vehicle Exposure",
    "B": "High Walk / High Vehicle Exposure",
    "C": "Low Walk / High Vehicle Exposure",
    "D": "Low Walk / Low Vehicle Exposure",
}

In [22]:
def flatten_frame_list(frame_list):
    frames = []

    for item in frame_list:
        if pd.isna(item):
            continue

        for fid in str(item).split(";"):
            fid = fid.strip()
            if fid:
                frames.append(int(fid))

    return sorted(set(frames))

def frame_path(fid):
    fname = f"{int(fid):010d}.png"
    return fname, os.path.join(img_dir, fname)

# Ensure quad_id exists
df_merge["quad_id"] = df_merge["quad_class"].astype(str).str.extract(r"^([A-D])")[0]

# Find quadrant(s) for one frame id
# def find_frame_quads(fid):
#     fid = str(int(fid))

#     matches = df_merge[
#         df_merge["frame_ids"]
#         .fillna("")
#         .astype(str)
#         .str.split(";")
#         .apply(lambda frames: fid in frames)
#     ]

#     return sorted(matches["quad_id"].dropna().unique())

# def show_frame_quad(fid):
#     fid = int(fid)
#     fname, fpath = frame_path(fid)
#     quads = find_frame_quads(fid)

#     print(f"Frame ID: {fid}")
#     print(f"File: {fname}")

#     if quads:
#         print("Quadrant label(s):")
#         for q in quads:
#             print(f"{q}: {quad_labels.get(q, 'Unknown')}")
#     else:
#         print("Quadrant label: Not found")

#     if os.path.exists(fpath):
#         display(Image.open(fpath))
#     else:
#         print(f"[MISSING] Expected file not found:\n{fpath}")


        
def find_frame_rows(fid):
    fid = str(int(fid))

    matches = df_merge[
        df_merge["frame_ids"]
        .fillna("")
        .astype(str)
        .str.split(";")
        .apply(lambda frames: fid in frames)
    ].copy()

    return matches

def show_frame_quad(fid):
    fid = int(fid)
    fname, fpath = frame_path(fid)
    matches = find_frame_rows(fid)

    print(f"Frame ID: {fid}")
    print(f"File: {fname}")

    if len(matches) > 0:
        quad_counts = matches["quad_id"].dropna().value_counts()

        dominant_quad = quad_counts.idxmax()
        dominant_rows = matches[matches["quad_id"] == dominant_quad]

        vehicle_exposure = pd.to_numeric(
            dominant_rows["veh_density_p90"],
            errors="coerce"
        ).median()

        composite_walkscore = pd.to_numeric(
            dominant_rows["walkscore_composite"],
            errors="coerce"
        ).median()

        print("Dominant quadrant:")
        print(f"{dominant_quad}: {quad_labels.get(dominant_quad, 'Unknown')}")
        print(f"Vehicle exposure: {vehicle_exposure:.3f}")
        print(f"Composite walkscore: {composite_walkscore:.3f}")

        print("\nMatching grid-cell quadrant counts:")
        print(quad_counts.to_string())
    else:
        print("Quadrant label: Not found in df_merge")
        print("Image exists, but this frame was not assigned to a quadrant.")

    if os.path.exists(fpath):
        display(Image.open(fpath))
    else:
        print(f"[MISSING] Expected file not found:\n{fpath}")

In [23]:
# Build quad_frames from quad_grouped
quad_col = "quad_id" if "quad_id" in quad_grouped.columns else "quad"

quad_frames = {}

for q in ["A", "B", "C", "D"]:
    row = quad_grouped.loc[quad_grouped[quad_col] == q, "frame_list"]

    if len(row) > 0:
        quad_frames[q] = flatten_frame_list(row.iloc[0])
    else:
        quad_frames[q] = []

def plot_quad_videos(quad_frames, fps=5, max_frames=100):
    n_frames = max(len(frames) for frames in quad_frames.values())

    if max_frames is not None:
        n_frames = min(n_frames, max_frames)

    for i in range(n_frames):
        clear_output(wait=True)

        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        axes = axes.ravel()

        for ax, q in zip(axes, ["A", "B", "C", "D"]):
            ax.axis("off")

            if len(quad_frames[q]) == 0:
                ax.set_title(f"{q}: {quad_labels[q]}\nNo frames")
                continue

            fid = quad_frames[q][min(i, len(quad_frames[q]) - 1)]
            fname, fpath = frame_path(fid)

            if os.path.exists(fpath):
                img = Image.open(fpath)
                ax.imshow(img)
                ax.set_title(f"{q}: {quad_labels[q]}\nFrame ID: {fid}")
            else:
                ax.set_title(f"{q}: {quad_labels[q]}\n[MISSING] {fname}")

        plt.tight_layout()
        display(fig)
        plt.close(fig)

def save_quad_video_gif(
    quad_frames,
    output_path="outputs/quad_quadrants_video.gif",
    fps=5,
    max_frames=100
):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    n_frames = max(len(frames) for frames in quad_frames.values())

    if max_frames is not None:
        n_frames = min(n_frames, max_frames)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.ravel()

    def update(i):
        for ax, q in zip(axes, ["A", "B", "C", "D"]):
            ax.clear()
            ax.axis("off")

            if len(quad_frames[q]) == 0:
                ax.set_title(f"{q}: {quad_labels[q]}\nNo frames")
                continue

            fid = quad_frames[q][min(i, len(quad_frames[q]) - 1)]
            fname, fpath = frame_path(fid)

            if os.path.exists(fpath):
                img = Image.open(fpath)
                ax.imshow(img)
                ax.set_title(f"{q}: {quad_labels[q]}\nFrame ID: {fid}")
            else:
                ax.set_title(f"{q}: {quad_labels[q]}\n[MISSING] {fname}")

        plt.tight_layout()

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=n_frames,
        interval=1000 / fps,
        blit=False
    )

    ani.save(output_path, writer="pillow", fps=fps, dpi=100)
    plt.close(fig)

    print(f"Saved GIF to: {output_path}")

In [ ]:
# ---------------- INTERFACE ----------------

frame_input = widgets.IntText(
    value=8585,
    description="Frame ID:"
)

show_button = widgets.Button(
    description="Show Frame",
    button_style="primary"
)

play_button = widgets.Button(
    description="Play Quadrants",
    button_style="success"
)

fps_input = widgets.IntText(
    value=5,
    description="FPS:"
)

max_frames_input = widgets.IntText(
    value=100,
    description="Max Frames:"
)

stop_button = widgets.Button(
    description="Stop Video",
    button_style="danger"
)

stop_flag = {"stop": False}



output = widgets.Output()

In [25]:
def on_show_clicked(_):
    with output:
        clear_output(wait=True)
        show_frame_quad(frame_input.value)

def on_play_clicked(_):
    with output:
        clear_output(wait=True)
        plot_quad_videos(
            quad_frames,
            fps=fps_input.value,
            max_frames=max_frames_input.value
        )
        
def on_stop_clicked(_):
    stop_flag["stop"] = True


show_button.on_click(on_show_clicked)
play_button.on_click(on_play_clicked)
stop_button.on_click(on_stop_clicked)

display(widgets.VBox([
    widgets.HBox([frame_input, show_button]),
    widgets.HBox([fps_input, max_frames_input, play_button, stop_button]),
    output
]))

In [26]:
save_quad_video_gif(
    quad_frames,
    output_path="outputs/quad_quadrants_video.gif",
    fps=5,
    max_frames=100
)

Saved GIF to: outputs/quad_quadrants_video.gif


In [27]:
missing_quad_rows = df_merge[
    df_merge["quad_id"].isna() | df_merge["quad_class"].isna()
]

print("Rows missing quad labels:", len(missing_quad_rows))
missing_quad_rows[["frame_ids", "quad_class", "quad_id"]].head()

Rows missing quad labels: 0


,frame_ids,quad_class,quad_id


In [28]:
frame_quad = []

for _, row in df_merge.iterrows():
    quad_id = row.get("quad_id")
    quad_class = row.get("quad_class")
    frame_ids = str(row.get("frame_ids", "")).split(";")

    for fid in frame_ids:
        fid = fid.strip()
        if fid:
            frame_quad.append({
                "frame_id": int(fid),
                "quad_id": quad_id,
                "quad_class": quad_class
            })

frame_quad_df = pd.DataFrame(frame_quad)

missing_frame_quads = frame_quad_df[
    frame_quad_df["quad_id"].isna() | frame_quad_df["quad_class"].isna()
]

print("Total unique frame ids:", frame_quad_df["frame_id"].nunique())
print("Frame ids missing quad labels:", missing_frame_quads["frame_id"].nunique())

missing_frame_quads.head()

Total unique frame ids: 3177
Frame ids missing quad labels: 0


,frame_id,quad_id,quad_class


In [29]:
frame_quad_df.groupby("quad_id")["frame_id"].nunique().reset_index(
    name="unique_frame_count"
)

,quad_id,unique_frame_count
0,A,2050
1,B,984
2,C,1861
3,D,1097


In [30]:
all_frame_ids = []

for ids in df_merge["frame_ids"].dropna().astype(str):
    for fid in ids.split(";"):
        fid = fid.strip()
        if fid:
            all_frame_ids.append(int(fid))

unique_frame_ids = sorted(set(all_frame_ids))

print("Total frame_id entries:", len(all_frame_ids))
print("Unique frame_ids contributed:", len(unique_frame_ids))
print("Min frame_id:", min(unique_frame_ids))
print("Max frame_id:", max(unique_frame_ids))

Total frame_id entries: 28117
Unique frame_ids contributed: 3177
Min frame_id: -1
Max frame_id: 11467


In [31]:
frame_counts_by_quad = []

for q, group in df_merge.groupby("quad_id"):
    frame_ids = []

    for ids in group["frame_ids"].dropna().astype(str):
        for fid in ids.split(";"):
            fid = fid.strip()
            if fid:
                frame_ids.append(int(fid))

    frame_counts_by_quad.append({
        "quad_id": q,
        "unique_frame_ids": len(set(frame_ids)),
        "total_frame_id_entries": len(frame_ids)
    })

pd.DataFrame(frame_counts_by_quad)

,quad_id,unique_frame_ids,total_frame_id_entries
0,A,2050,11339
1,B,984,3650
2,C,1861,10572
3,D,1097,2556


In [32]:
df_merge["veh_density_p90"] 

0       0.333333
1       0.333333
2       0.666667
3       0.666667
4       0.666667
          ...   
1383    0.333333
1384    0.000000
1385    0.069152
1386    0.217864
1387    0.333333
Name: veh_density_p90, Length: 1388, dtype: float64